In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
import os
import json
from PIL import Image

# -----------------------------
# CONFIG
# -----------------------------
KITTI_ROOT = os.path.join("data", "Kitty")
IMAGE_DIR = os.path.join(KITTI_ROOT, "data_object_image_2", "training", "image_2")
LABEL_DIR = os.path.join(KITTI_ROOT, "data_object_label_2", "training", "label_2")
OUTPUT_JSON = os.path.join("data", "Kitty", "kitti_coco.json")

# Keep only these classes
VALID_CLASSES = {
    "Car": 3,
    "Pedestrian": 1
}

# -----------------------------
# HELPERS
# -----------------------------
def parse_kitti_label_file(label_path):
    """
    Parses a KITTI label file.
    Returns a list of valid annotations.
    """
    annotations = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()

        cls = parts[0]
        truncation = float(parts[1])
        occlusion = int(parts[2])

        # I just care about this classes, non-truncated and fully-visible
        if cls not in VALID_CLASSES or occlusion != 0 or truncation >= 0.1:
            continue

        # KITTI bbox format:
        # parts[4:8] = xmin, ymin, xmax, ymax
        xmin = float(parts[4])
        ymin = float(parts[5])
        xmax = float(parts[6])
        ymax = float(parts[7])

        width = xmax - xmin
        height = ymax - ymin

        annotations.append({
            "category_id": VALID_CLASSES[cls],
            "bbox": [xmin, ymin, width, height],
            "area": width * height,
            "iscrowd": 0
        })

    return annotations


# -----------------------------
# MAIN CONVERSION
# -----------------------------
def convert():
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 1, "name": "Car"},
            {"id": 2, "name": "Pedestrian"}
        ]
    }

    image_id = 0
    annotation_id = 0

    image_files = sorted(os.listdir(IMAGE_DIR))

    for img_name in image_files:
        if not img_name.endswith((".png", ".jpg", ".jpeg")):
            continue

        img_path = os.path.join(IMAGE_DIR, img_name)
        label_path = os.path.join(LABEL_DIR, img_name.replace(".png", ".txt"))

        if not os.path.exists(label_path):
            continue

        anns = parse_kitti_label_file(label_path)

        # Skip images with no valid objects
        if len(anns) == 0:
            continue

        # Load image to get dimensions
        with Image.open(img_path) as img:
            width, height = img.size

        # Add image entry
        coco_output["images"].append({
            "id": image_id,
            "file_name": img_name,
            "width": width,
            "height": height
        })

        # Add annotations
        for ann in anns:
            ann["id"] = annotation_id
            ann["image_id"] = image_id

            coco_output["annotations"].append(ann)

            annotation_id += 1

        image_id += 1

    # Save JSON
    with open(OUTPUT_JSON, "w") as f:
        json.dump(coco_output, f, indent=4)

    print(f"Conversion complete. Saved to {OUTPUT_JSON}")
    print(f"Images: {len(coco_output['images'])}")
    print(f"Annotations: {len(coco_output['annotations'])}")


if __name__ == "__main__":
    if not os.path.exists(OUTPUT_JSON):
        convert()

In [ ]:
from detectron2.data.datasets.coco_scale import KITTICocoDataset
debug = True
kitty_train = KITTICocoDataset(
    OUTPUT_JSON, 
    IMAGE_DIR
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

if debug:
    max_vis = 10
    for i, d in enumerate(kitty_train):
        print(d["file_name"])
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5)
        visualizer.draw_dataset_dict(d)
        out = visualizer.get_output()
        img = out.get_image()
        plt.imshow(img)
        plt.show()
        if max_vis == i:
            break

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets.builtin_meta import _get_builtin_metadata
kitty_dataset_name = "kitty_train"

DatasetCatalog.register(
    kitty_dataset_name,
    KITTICocoDataset(
        OUTPUT_JSON,
        IMAGE_DIR,
    )
)

coco_meta = _get_builtin_metadata("coco")
MetadataCatalog.get(kitty_dataset_name).set(
    thing_dataset_id_to_contiguous_id = {1: 0, 3:2}  # COCO ID 1 → internal ID 0
)

In [ ]:
import os
from detectron2.engine import DefaultTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.6,max_split_size_mb:128,expandable_segments:True"

cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
# cfg.DATALOADER
cfg.DATASETS.TRAIN = (kitty_dataset_name, )
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 16  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.02  # pick a good LR
cfg.MODEL.KEYPOINT_ON = False
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2  # Number of classes
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 17 # Number of keypoints
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = True  # Althought must be set to false in joined dataset
cfg.DATALOADER.ASPECT_RATIO_GROUPING = True  # Althought must be set to false in joined dataset
cfg.SOLVER.MAX_ITER = 500
experiment_name = "test-debug-calib"
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.
cfg.FLOAT32_PRECISION = "medium"

trainer = DefaultTrainer(cfg) 
# NOTE: change value of resume if we have a last_checkpoint
trainer.resume_or_load(resume=False)
trainer.train()